# Mini-Projeto 01 - Análise Exploratória de Dados no Varejo

## Introdução

Este mini-projeto tem como objetivo realizar uma Análise Exploratória de Dados (AED) utilizando uma base de varejo. A proposta é transformar dados brutos de compras em informações organizadas, identificando características da base, possíveis problemas de qualidade e padrões relevantes sobre clientes, produtos e categorias.

A base analisada contém registros de compras com informações como data da transação, identificação do cliente, gênero, estado civil, número de filhos, segmento do cliente, identificação do produto, categoria e nome do produto. A partir desses dados, será possível praticar etapas importantes de um projeto de análise de dados, como carregamento da base, verificação dos tipos de dados, tratamento de valores nulos, remoção de colunas irrelevantes, conversão de datas e criação de estatísticas descritivas.

Além da limpeza e preparação dos dados, o projeto também busca responder perguntas operacionais simples, como quais categorias aparecem com maior frequência, quais perfis de clientes concentram mais registros de compra e como os dados estão distribuídos ao longo do tempo. Essas análises ajudam a compreender melhor o comportamento da base e servem como ponto de partida para visualizações, relatórios ou dashboards.

Ao final, espera-se apresentar os principais insights encontrados e registrar eventuais limitações ou problemas remanescentes nos dados, comunicando os resultados de forma clara e objetiva.


In [ ]:
# Importa bibliotecas auxiliares, realiza o download do dataset do Kaggle e copia os arquivos para a pasta local data/

import kagglehub
import shutil
from pathlib import Path

# Download latest version
path = kagglehub.dataset_download("namespaiva/base-varejo")

print("Dataset baixado em:", path)

# Pasta de destino do projeto

destino = Path("./data")

destino.mkdir(exist_ok=True)

# Copiar os arquivos para a pasta de destino
for arquivo in Path(path).glob("*"):
    destino_arquivo = destino / arquivo.name

    if not destino_arquivo.exists():
        shutil.copy(arquivo, destino_arquivo)

print("Arquivos copiados para:", destino)



In [ ]:
# Importação da biblioteca pandas
import pandas as pd

# Carrega o dataset para o ambiente de desenvolvimento
df = pd.read_csv(destino / "Base Varejo.csv", sep=";", encoding="cp1252")

print("Dataset carregado com sucesso!\n")
print(f"O dataframe tem {df.shape[0]} linhas e {df.shape[1]} colunas.\n")

# Exibe as primeiras linhas do dataset
print(df.head().to_string())
print("\nColunas do dataset:", df.columns.tolist())
print("\nTipos de dados das colunas:\n", df.dtypes)


In [ ]:
# ANÁLISE DA BASE DE DADOS

# Estatísticas descritivas
print("\nEstatísticas descritivas do dataset:\n")
print(df.describe(include="all").round(2).to_string())

# Quantidade de nulos por coluna
print("\nQuantidade de valores nulos por coluna:\n")
print(df.isnull().sum())

# Verificar valores únicos 
print("\nValores únicos por coluna:\n")
for coluna in df.columns:
    print(f"{coluna}: {df[coluna].nunique()} valores únicos")

# Quantidade de linhas duplicadas

linhas_duplicadas = df.duplicated().sum()
print(f"\nQuantidade de linhas duplicadas: {linhas_duplicadas}")



### Problemas encontrados na base

Após a verificação inicial, foram identificados alguns pontos de atenção:

- As colunas `Unnamed: 10`, `Unnamed: 11`, `Unnamed: 12` e `Unnamed: 13` possuem todos os valores nulos, indicando que não trazem informação útil para a análise.
- A base foi verificada quanto à existência de linhas duplicadas.
- A coluna `DATA` foi testada para identificar possíveis datas inválidas.
- A coluna `PR_CAT` apresenta a categoria `#N/D`, que indica registros sem categoria de produto bem definida.

In [ ]:
# LIMPEZA E TRATAMENTO DOS DADOS

# Remover colunas irrelevantes
colunas_irrelevantes = ['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']
df_limpo = df.copy()

df_limpo = df_limpo.drop(columns=colunas_irrelevantes)

print("Colunas removidas por conterem apenas valores nulos:\n")
print(colunas_irrelevantes)

# Converter coluna data para o formato datetime
df_limpo['DATA'] = pd.to_datetime(
    df_limpo['DATA'], 
    errors='coerce',
    format='%d/%m/%Y'
    )

# Verificar o número de datas inválidas após a conversão
datas_invalidas = df_limpo['DATA'].isnull().sum()
print(f"\nNúmero de datas inválidas após a conversão: {datas_invalidas}")

# Remover linhas duplicadas
linhas_duplicadas_antes_limpeza = df_limpo.duplicated().sum()

df_limpo = df_limpo.drop_duplicates()

linhas_duplicadas_apos_limpeza = df_limpo.duplicated().sum()

print(f"\nQuantidade de linhas duplicadas antes da limpeza: {linhas_duplicadas_antes_limpeza}")
print(f"Quantidade de linhas duplicadas após a limpeza: {linhas_duplicadas_apos_limpeza}")
print(f"Formato da base após remoção de duplicatas: {df_limpo.shape}\n")

# Tratamento de categoria inconsistente da coluna PR_CAT

df_limpo["PR_CAT"] = df_limpo["PR_CAT"].replace("#N/D", "SEM CATEGORIA")

print(df_limpo["PR_CAT"].value_counts())



As colunas `Unnamed: 10`, `Unnamed: 11`, `Unnamed: 12` e `Unnamed: 13` foram removidas porque possuem 100% dos valores nulos. Como essas colunas não apresentam informação útil para a análise, a melhor escolha foi removê-las em vez de imputar valores.

As linhas duplicadas foram removidas para evitar contagens repetidas nas análises. Esse tratamento é importante porque registros duplicados podem distorcer resultados como quantidade de compras, produtos mais frequentes e análises por categoria.

A coluna `DATA` foi convertida para o tipo `datetime`, permitindo análises temporais. 

Na coluna `PR_CAT`, o valor `#N/D` foi substituído por `SEM CATEGORIA`, pois indica produtos sem categoria definida. A opção foi manter esses registros na base, já que eles ainda podem conter informações úteis sobre clientes, datas e produtos.

In [ ]:
# TRATAMENTO E EXPLORAÇÃO DE DADOS CATEGÓRICOS

# Criar colunas categóricas a partir da coluna 'CL_EC'
estado_civil_map = {
    1: "Casado/União Estável",
    2: "Divorciado",
    3: "Separado",
    4: "Solteiro",
    5: "Viúvo"
}
df_limpo['CL_EC_CATEGORIA'] = df_limpo['CL_EC'].map(estado_civil_map)

# Transformar colunas string para categóricas
colunas_categoricas = ['CL_EC_CATEGORIA', 'CL_GENERO', 'CL_SEG', 'PR_CAT', 'PR_NOME']
for coluna in colunas_categoricas:
    df_limpo[coluna] = df_limpo[coluna].astype('category')

# Verificar se a conversão foi bem-sucedida
print("\nTipos de dados após conversão:", df_limpo.dtypes)

# Explorando colunas categóricas

for coluna in colunas_categoricas:
    print(f"\nValores únicos na coluna (primeiros 10) '{coluna}':\n")
    print(df_limpo[coluna].value_counts().head(10).to_string())


Nesta etapa, as variáveis categóricas da base foram preparadas para facilitar a análise. A coluna `CL_EC`, que representa o estado civil do cliente por meio de códigos numéricos, foi transformada em uma nova coluna com descrições mais claras.

Também foram convertidas para o tipo `category` as colunas que representam grupos ou classificações, como gênero, segmento do cliente, categoria do produto e nome do produto. Essa conversão melhora a organização dos dados e facilita a exploração das frequências de cada categoria.

Por fim, foram exibidos os principais valores de cada coluna categórica para entender quais grupos aparecem com maior frequência na base de varejo.